In [1]:
import slangpy as spy
from pyglm import glm
import matplotlib.pyplot as plt
import numpy as np

from bvhgs import device
from bvhgs.camera import Camera
from bvhgs.gaussian import GaussianCloud
from bvhgs.renderer import Renderer

[INFO] (rhi) layer: CreateDevice: Debug layer is enabled.
[WARN] No supported shader model found, pretending to support sm_6_0.


In [2]:
np.random.seed(348)

# Create Gaussian Buffer

In [3]:
gaussians = GaussianCloud(16)
len(gaussians)

16

# Load Module and Shader

In [4]:
module = device.load_module("renderer.slang")
module

SlangModule(
  name = renderer.slang,
  path = /Users/fangjun/Documents/stanford/bvhgs/src/bvhgs/slang/renderer.slang,
  entry_points = [
    SlangEntryPoint(name="project", stage=compute),
    SlangEntryPoint(name="cull", stage=compute),
    SlangEntryPoint(name="rasterize", stage=compute),
  ]
)

In [5]:
program = device.link_program([module], [])
program

ShaderProgram(
  modules = [
    SlangModule(
      name = renderer.slang,
      path = /Users/fangjun/Documents/stanford/bvhgs/src/bvhgs/slang/renderer.slang,
      entry_points = [
        SlangEntryPoint(name="project", stage=compute),
        SlangEntryPoint(name="cull", stage=compute),
        SlangEntryPoint(name="rasterize", stage=compute),
      ]
    ),
  ],
  entry_points = []
)

# Projection

## Build Slang Buffer and Camera Parameter

In [6]:
# Create a buffer for the Gaussian points.
gaussian_buf = device.create_buffer(
    element_count=len(gaussians),
    struct_type=program.reflection.g_gaussian_3d,
    usage=spy.BufferUsage.shader_resource,
)
# Store all the gaussian points in the buffer.
gaussian_cursor = spy.BufferCursor(
    program.reflection.g_gaussian_3d.type_layout.element_type_layout,
    gaussian_buf,
)
for i in range(len(gaussians)):
    print(gaussians[i]["position"])
    gaussian_cursor[i].write(gaussians[i])
gaussian_cursor.apply()

[0.27285782 0.3842838  0.33193865]
[0.9735973 0.446306  0.7633389]
[0.57010466 0.7979699  0.50363845]
[0.8289161  0.04957912 0.6569258 ]
[0.34960684 0.19731732 0.07787938]
[0.78357005 0.39515856 0.66507125]
[0.06987321 0.13599738 0.12401959]
[0.21811415 0.8226211  0.904117  ]
[0.18196344 0.41448593 0.7727715 ]
[0.1977999  0.3255716  0.58641833]
[0.22465152 0.9372966  0.05132372]
[0.97091204 0.9751557  0.91381645]
[0.09896851 0.67623353 0.23060538]
[0.6377924 0.7643661 0.7383538]
[0.11066168 0.9894526  0.22375931]
[0.28214398 0.671215   0.84969634]


In [7]:
# Create a buffer for the Gaussian2D points.
gaussian2d_buf = device.create_buffer(
    element_count=len(gaussians),
    struct_type=program.reflection.g_gaussian_2d,
    usage=spy.BufferUsage.shader_resource | spy.BufferUsage.unordered_access,
)
gaussian2d_cursor = spy.BufferCursor(
    program.reflection.g_gaussian_2d.type_layout.element_type_layout,
    gaussian2d_buf,
)
gaussian2d_cursor[0].read()

{'position': {0, 0, 0},
 'covariance': {{0, 0}, {0, 0}},
 'color': {0, 0, 0},
 'opacity': 0.0,
 'cachedInvCov': {{0, 0}, {0, 0}},
 'cachedDet': 0.0,
 'cachedNorm': 0.0}

In [8]:
camera = Camera(
    rotation=glm.quat(1, 0, 0, 0),
    translation=glm.vec3(0, 0, 0),
    sensor_size=glm.uvec2(512, 512),
    focal_length=64
)
camera.to_slang()

{'_rotation': [0.0, 0.0, 0.0, 1.0],
 '_translation': vec3( 0, 0, 0 ),
 '_sensorSize': uvec2( 512, 512 ),
 '_focalLength': 64}

## Dispatch Projection Kernel

In [9]:
# Create flag buffer for culling.
cull_flag_buf = device.create_buffer(
    element_count=len(gaussians),
    struct_type=program.reflection.g_cull_flag,
    usage=spy.BufferUsage.shader_resource | spy.BufferUsage.unordered_access,
)

In [10]:
ker_proj = device.create_compute_kernel(
    device.link_program([module], [module.entry_point("project")])
)
ker_proj

ComputeKernel(0x6000014f9f00)

In [11]:
ker_proj.dispatch(
    thread_count=[len(gaussians), 1, 1],
    vars={
        "g_camera": camera.to_slang(),
        "g_gaussian_3d": gaussian_buf,
        "g_gaussian_2d": gaussian2d_buf,
        "g_cull_flag": cull_flag_buf
    }
)

In [12]:
gaussian2d_cursor = spy.BufferCursor(
    program.reflection.g_gaussian_2d.type_layout.element_type_layout,
    gaussian2d_buf,
)
for i in range(gaussian2d_cursor.element_count):
    print(gaussian2d_cursor[i].read()["position"])

{0.8220128, 1.1576953, 0.57646227}
{1.2754456, 0.584676, 1.315206}
{1.1319721, 1.5844102, 1.102464}
{1.2618108, 0.07547142, 1.0588255}
{4.4890814, 2.5336275, 0.40893063}
{1.1781746, 0.5941597, 1.101114}
{0.5634046, 1.0965798, 0.19687156}
{0.2412455, 0.9098614, 1.2416549}
{0.23546864, 0.5363629, 0.89559203}
{0.3373017, 0.5551866, 0.69929117}
{4.3771477, 18.262444, 0.96520853}
{1.0624804, 1.0671244, 1.6518654}
{0.42916828, 2.9324274, 0.7212943}
{0.86380327, 1.0352302, 1.2394357}
{0.49455678, 4.421951, 1.0204561}
{0.33205274, 0.789947, 1.1189811}


## Cull Gaussians

In [13]:
prefix_sum = cull_flag_buf.to_numpy()
prefix_sum = np.cumsum(prefix_sum).astype(np.uint32)
print(prefix_sum)

[0 0 0 0 0 0 0 1 2 3 3 3 3 3 3 4]


In [14]:
num_culled_gaussians = prefix_sum[-1]
print(num_culled_gaussians)
cull_prefix_buf = device.create_buffer(
    element_count=len(gaussians),
    struct_type=program.reflection.g_cull_prefix,
    usage=spy.BufferUsage.shader_resource | spy.BufferUsage.unordered_access,
)
cull_prefix_buf.copy_from_numpy(prefix_sum)

4


In [15]:
# Culled gaussians
culled_gaussian_buf = device.create_buffer(
    element_count=num_culled_gaussians,
    struct_type=program.reflection.g_gaussian_2d_culled,
    usage=spy.BufferUsage.shader_resource | spy.BufferUsage.unordered_access,
)

In [16]:
ker_cull = device.create_compute_kernel(
    device.link_program([module], [module.entry_point("cull")])
)
ker_cull.dispatch(
    thread_count=[len(gaussians), 1, 1],
    vars={
        "g_gaussian_2d": gaussian2d_buf,
        "g_cull_flag": cull_flag_buf,
        "g_cull_prefix": cull_prefix_buf,
        "g_gaussian_2d_culled": culled_gaussian_buf
    }
)

In [17]:
gaussian2d_cursor = spy.BufferCursor(
    program.reflection.g_gaussian_2d_culled.type_layout.element_type_layout,
    culled_gaussian_buf,
)
for i in range(gaussian2d_cursor.element_count):
    print(gaussian2d_cursor[i].read()["position"])

{0.2412455, 0.9098614, 1.2416549}
{0.23546864, 0.5363629, 0.89559203}
{0.3373017, 0.5551866, 0.69929117}
{0.33205274, 0.789947, 1.1189811}
